In [ ]:
!pip install -q accelerate datasets transformers scikit-learn pandas numpy torch


# Multilingual Prompt-Attack Classification

**Chosen model:** `distilbert-base-multilingual-cased` (DistilBERT multilingual, cased)


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, DataCollatorWithPadding
from datasets import Dataset

MODEL_NAME = 'distilbert-base-multilingual-cased'
LABELS = ['BENIGN', 'JAILBREAK', 'PROMPT_INJECTION']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}


In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
train_df['label_id'] = train_df['label'].map(label2id)
print(train_df['label'].value_counts())
print(train_df['language'].value_counts())


In [ ]:
tr, va = train_test_split(train_df.sample(2000, random_state=42), test_size=0.1, stratify=train_df.sample(2000, random_state=42)['label_id'], random_state=42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=64)

tr_ds = Dataset.from_pandas(tr[['text','label_id']].rename(columns={'label_id':'labels'}), preserve_index=False)
va_ds = Dataset.from_pandas(va[['text','label_id']].rename(columns={'label_id':'labels'}), preserve_index=False)
te_ds = Dataset.from_pandas(test_df[['text']], preserve_index=False)
tr_ds = tr_ds.map(tokenize, batched=True)
va_ds = va_ds.map(tokenize, batched=True)
te_ds = te_ds.map(tokenize, batched=True)


In [ ]:
classes = np.unique(tr['label_id'])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=tr['label_id'].values)
class_weights = torch.tensor(weights, dtype=torch.float)
print('class_weights', class_weights)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id)


In [ ]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir='./out',
    num_train_epochs=1,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=100,
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tr_ds,
    eval_dataset=va_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
trainer.train()


In [ ]:
preds = trainer.predict(te_ds)
pred_ids = np.argmax(preds.predictions, axis=1)
pred_labels = [id2label[i] for i in pred_ids]
submission = pd.DataFrame({'id': test_df['id'], 'label': pred_labels})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print(submission['label'].value_counts())
